In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Ma trận: hàng = sinh viên, cột = sự kiện, giá trị = 1 (đã đăng ký) hoặc 0
ratings_data = {
    'SV001': [1, 1, 1, 0, 0, 1, 1],
    'SV002': [1, 0, 1, 0, 0, 1, 0],
    'SV003': [0, 1, 0, 1, 1, 0, 1],
    'SV004': [0, 1, 0, 1, 1, 0, 1],
    'SV005': [1, 1, 1, 0, 0, 0, 1],
    'SV006': [0, 0, 0, 1, 1, 0, 0],
}

event_names = [
    'E1_HienMau', 'E2_KyNangMem', 'E3_MoiTruong',
    'E4_LapTrinh', 'E5_HuongNghiep', 'E6_HienMau2', 'E7_ThuyetTrinh'
]

df_ratings = pd.DataFrame(ratings_data, index=event_names).T
print("Ma trận đăng ký sinh viên - sự kiện:")
print(df_ratings)

Ma trận đăng ký sinh viên - sự kiện:
       E1_HienMau  E2_KyNangMem  E3_MoiTruong  E4_LapTrinh  E5_HuongNghiep  \
SV001           1             1             1            0               0   
SV002           1             0             1            0               0   
SV003           0             1             0            1               1   
SV004           0             1             0            1               1   
SV005           1             1             1            0               0   
SV006           0             0             0            1               1   

       E6_HienMau2  E7_ThuyetTrinh  
SV001            1               1  
SV002            1               0  
SV003            0               1  
SV004            0               1  
SV005            0               1  
SV006            0               0  


In [7]:
# Tính cosine similarity giữa các sinh viên (user-user)
user_sim_matrix = cosine_similarity(df_ratings)
df_user_sim = pd.DataFrame(user_sim_matrix, 
                            index=df_ratings.index, 
                            columns=df_ratings.index)

print("Ma trận độ tương đồng giữa sinh viên:")
print(df_user_sim.round(3))



Ma trận độ tương đồng giữa sinh viên:
       SV001  SV002  SV003  SV004  SV005  SV006
SV001  1.000  0.775  0.447  0.447  0.894  0.000
SV002  0.775  1.000  0.000  0.000  0.577  0.000
SV003  0.447  0.000  1.000  1.000  0.500  0.707
SV004  0.447  0.000  1.000  1.000  0.500  0.707
SV005  0.894  0.577  0.500  0.500  1.000  0.000
SV006  0.000  0.000  0.707  0.707  0.000  1.000


In [8]:
def goi_y_collaborative(student_id, top_n=3, n_similar_users=2):
    """
    Gợi ý sự kiện cho student_id dựa trên sinh viên tương tự
    """
    # Lấy sinh viên tương tự nhất (bỏ chính sinh viên đó)
    sim_scores = df_user_sim[student_id].drop(student_id).sort_values(ascending=False)
    top_similar_users = sim_scores.head(n_similar_users).index.tolist()
    
    print(f"  → Sinh viên tương tự nhất với {student_id}: {top_similar_users}")
    
    # Sự kiện sinh viên hiện tại CHƯA đăng ký
    da_dang_ky = df_ratings.loc[student_id]
    chua_dang_ky = da_dang_ky[da_dang_ky == 0].index.tolist()
    
    # Tính điểm cho từng sự kiện chưa đăng ký
    scores = {}
    for event in chua_dang_ky:
        score = 0
        for similar_user in top_similar_users:
            # Trọng số = similarity * (đã đăng ký hay chưa)
            score += df_user_sim.loc[student_id, similar_user] * df_ratings.loc[similar_user, event]
        scores[event] = score
    
    # Sắp xếp và trả về top_n
    result = pd.Series(scores).sort_values(ascending=False).head(top_n)
    return result

# Test
print("="*55)
print(f"📌 Gợi ý cho SV001 (chuyên tình nguyện):")
rec = goi_y_collaborative('SV001', top_n=3)
print(f"\n  → Sự kiện gợi ý (điểm càng cao = càng phù hợp):")
print(rec)


📌 Gợi ý cho SV001 (chuyên tình nguyện):
  → Sinh viên tương tự nhất với SV001: ['SV005', 'SV002']

  → Sự kiện gợi ý (điểm càng cao = càng phù hợp):
E4_LapTrinh       0.0
E5_HuongNghiep    0.0
dtype: float64


In [9]:
for sv in ['SV001', 'SV003', 'SV006']:
    print(f"\n{'='*55}")
    print(f"📌 Gợi ý cho {sv}:")
    goi_y_collaborative(sv, top_n=2)


📌 Gợi ý cho SV001:
  → Sinh viên tương tự nhất với SV001: ['SV005', 'SV002']

📌 Gợi ý cho SV003:
  → Sinh viên tương tự nhất với SV003: ['SV004', 'SV006']

📌 Gợi ý cho SV006:
  → Sinh viên tương tự nhất với SV006: ['SV003', 'SV004']
